# Data Description
- Background: 簡述資料內容。若是新接收到的資料，請說明為何會收到這份資料，要分析什麼。
- date received: YYYYMMDD
- Path to data file: 原始資料在資料中心（如行資）之路徑。請填寫原始檔案之路徑。
- Unit of observation:
- Sample period:
- Known issues:
    -
    -
- Definition for each variable:
    -
    -
- Data path: `data/raw/<INPUT_FILE_NAME.csv>` (exact local project-relative file, separate from the source location above)
- Role: `raw`
- Verification status: `<pending; record the actual read/check results after execution>`
- SHA-256: `<pending; fill from the actual file, not a copied example>`
- Next processing notebook: `data/code/<PROCESSING_NAME.ipynb-or-Rmd>` (pending until created)

This notebook documents the unchanged raw file. It does not clean or overwrite it.
Paths below resolve from the project root or from `data/raw/`, `data/code/`, and
`data/processed/`. Replace all placeholders and choose reader options for the
actual format, encoding, IDs, dates, and missing-value codes before execution.


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


## Change for Your Own Data

Use the exact project-relative Data path above. A missing path stops execution;
do not substitute another CSV or use the template/example data. The CSV reader
below is a scaffold: configure dtypes, date parsing, missing codes, and the reader
for the actual stored format. Fill the expected SHA-256 from the exact file
inspected while preparing this notebook; placeholders or a changed file stop
data loading. Documentation stays pending until executed.


In [ ]:
INPUT_RELATIVE_PATH = "data/raw/<INPUT_FILE_NAME.csv>"
input_data_file = project_file(INPUT_RELATIVE_PATH, "raw")
EXPECTED_SHA256 = "<INSPECTED_INPUT_SHA256>"  # Fill from the exact inspected raw file.


## Summary


In [ ]:
input_fingerprint = verify_input_sha256(input_data_file, EXPECTED_SHA256)
df = pd.read_csv(input_data_file)  # Configure dtype/parse_dates/na_values for the actual file.
print({"data_path": INPUT_RELATIVE_PATH, "rows": len(df),
       "schema": {column: str(dtype) for column, dtype in df.dtypes.items()},
       "sha256": sha256_file(input_data_file)})


### Sample


In [ ]:
df.head(10)


### Summary Stats


In [ ]:
df.describe(include="all")
